# Battery Optimizer – Real SE3 Prices + Degradation Test

This notebook contains only the essential steps for the current test:

1. Fetch one day of real SE3 day-ahead prices
2. Define the battery optimizer
3. Run it without degradation cost
4. Run it with degradation cost
5. Compare actions, profit, and plots


In [1]:
import requests
import pandas as pd
import pulp
import plotly.express as px
import plotly.graph_objects as go


## 1. Fetch real SE3 prices

In [2]:
url = "https://www.elprisetjustnu.se/api/v1/prices/2026/09-14_SE3.json"

response = requests.get(url)
response.raise_for_status()

prices_df = pd.DataFrame(response.json())
prices_df.head()


,SEK_per_kWh,EUR_per_kWh,EXR,time_start,time_end
0,1.31955,0.11728,11.251285,2026-09-14T00:00:00+02:00,2026-09-14T00:15:00+02:00
1,1.32236,0.11753,11.251285,2026-09-14T00:15:00+02:00,2026-09-14T00:30:00+02:00
2,1.32394,0.11767,11.251285,2026-09-14T00:30:00+02:00,2026-09-14T00:45:00+02:00
3,1.32450,0.11772,11.251285,2026-09-14T00:45:00+02:00,2026-09-14T01:00:00+02:00
4,1.32416,0.11769,11.251285,2026-09-14T01:00:00+02:00,2026-09-14T01:15:00+02:00


In [3]:
buy_price_real = prices_df["SEK_per_kWh"].to_numpy()
sell_price_real = buy_price_real.copy()  # Simplification for this test

print("Number of intervals:", len(buy_price_real))
print("Expected for 15-min data:", 96)


Number of intervals: 96
Expected for 15-min data: 96


## 2. Battery parameters

For this test, buy and sell prices are identical. Later we will make them realistic and different.

In [4]:
capacity = 4.0              # kWh
initial_energy = 2.0         # kWh
min_energy = 0.0             # kWh
max_energy = 4.0             # kWh

max_charge_power = 2.0       # kW
max_discharge_power = 2.0    # kW

eta_charge = 0.95
eta_discharge = 0.95

dt = 0.25                    # 15 minutes = 0.25 h


## 3. Optimizer function

In [26]:
def optimize_battery(
    buy_price,
    sell_price,
    capacity,
    initial_energy,
    min_energy,
    max_energy,
    max_charge_power,
    max_discharge_power,
    eta_charge,
    eta_discharge,
    dt,
    degradation_cost=0.0
):
    T = len(buy_price)

    model = pulp.LpProblem("Battery_Optimizer", pulp.LpMaximize)

    charge = pulp.LpVariable.dicts(
        "charge", range(T), lowBound=0, upBound=max_charge_power
    )

    discharge = pulp.LpVariable.dicts(
        "discharge", range(T), lowBound=0, upBound=max_discharge_power
    )

    energy = pulp.LpVariable.dicts(
        "energy", range(T + 1), lowBound=min_energy, upBound=max_energy
    )

    # Objective
    model += pulp.lpSum(
        (
            sell_price[t] * discharge[t]
            - buy_price[t] * charge[t]
            - degradation_cost * discharge[t]
        ) * dt
        for t in range(T)
    )

    # Initial condition
    model += energy[0] == initial_energy

    # Battery dynamics
    for t in range(T):
        model += (
            energy[t + 1]
            ==
            energy[t]
            + eta_charge * charge[t] * dt
            - discharge[t] * dt / eta_discharge
        )

    # Terminal condition
    model += energy[T] == initial_energy

    # Solve
    model.solve(pulp.GLPK_CMD(msg=False))

    if pulp.LpStatus[model.status] != "Optimal":
        raise RuntimeError(
            f"Solver status: {pulp.LpStatus[model.status]}"
        )

    rows = []

    cumulative_trading_profit = 0.0
    cumulative_degradation_cost = 0.0
    cumulative_net_profit = 0.0

    for t in range(T):
        charge_t = charge[t].value()
        discharge_t = discharge[t].value()
        energy_t = energy[t].value()
        energy_after_t = energy[t + 1].value()

        trading_cashflow_t = (
            sell_price[t] * discharge_t
            - buy_price[t] * charge_t
        ) * dt

        degradation_cost_t = (
            degradation_cost * discharge_t
        ) * dt

        net_cashflow_t = (
            trading_cashflow_t
            - degradation_cost_t
        )

        cumulative_trading_profit += trading_cashflow_t
        cumulative_degradation_cost += degradation_cost_t
        cumulative_net_profit += net_cashflow_t

        if charge_t > 1e-6:
            action = "charge"
        elif discharge_t > 1e-6:
            action = "discharge"
        else:
            action = "hold"

        rows.append({
            "time": t,
            "buy_price": buy_price[t],
            "sell_price": sell_price[t],
            "charge_kw": charge_t,
            "discharge_kw": discharge_t,
            "energy_kwh": energy_t,
            "energy_after_kwh": energy_after_t,
            "soc": energy_t / capacity,
            "soc_after": energy_after_t / capacity,
            "action": action,
            "trading_cashflow_sek": trading_cashflow_t,
            "degradation_cost_sek": degradation_cost_t,
            "net_cashflow_sek": net_cashflow_t,
            "cumulative_trading_profit_sek": cumulative_trading_profit,
            "cumulative_degradation_cost_sek": cumulative_degradation_cost,
            "cumulative_net_profit_sek": cumulative_net_profit
        })

    return pd.DataFrame(rows)

## 4. Run without degradation cost

In [27]:
result_no_deg = optimize_battery(
    buy_price=buy_price_real,
    sell_price=sell_price_real,
    capacity=capacity,
    initial_energy=initial_energy,
    min_energy=min_energy,
    max_energy=max_energy,
    max_charge_power=max_charge_power,
    max_discharge_power=max_discharge_power,
    eta_charge=eta_charge,
    eta_discharge=eta_discharge,
    dt=dt,
    degradation_cost=0.0
)


## 5. Run with degradation cost

In [42]:
degradation_cost = 0.20  # SEK/kWh discharged

result_deg = optimize_battery(
    buy_price=buy_price_real,
    sell_price=sell_price_real,
    capacity=capacity,
    initial_energy=initial_energy,
    min_energy=min_energy,
    max_energy=max_energy,
    max_charge_power=max_charge_power,
    max_discharge_power=max_discharge_power,
    eta_charge=eta_charge,
    eta_discharge=eta_discharge,
    dt=dt,
    degradation_cost=degradation_cost
)


## 6. Add timestamps

In [30]:
for df_ in [result_no_deg, result_deg]:
    df_["time_start"] = pd.to_datetime(prices_df["time_start"])
    df_["time_end"] = pd.to_datetime(prices_df["time_end"])


## 7. Compare behavior

In [32]:
print("WITHOUT degradation")
print(result_no_deg["action"].value_counts())
print(
    "Net profit:",
    result_no_deg["cumulative_net_profit_sek"].iloc[-1]
)

print("\nWITH degradation")
print(result_deg["action"].value_counts())
print(
    "Net profit:",
    result_deg["cumulative_net_profit_sek"].iloc[-1]
)

WITHOUT degradation
action
hold         46
charge       27
discharge    23
Name: count, dtype: int64
Net profit: 9.922744440048792

WITH degradation
action
hold         63
charge       18
discharge    15
Name: count, dtype: int64
Net profit: 8.09972830729743


## 8. Price plot

In [33]:
fig_price = px.line(
    result_no_deg,
    x="time_start",
    y="buy_price",
    title="SE3 day-ahead price"
)
fig_price.show()


## 9. Charge/discharge comparison

In [34]:
fig_actions = go.Figure()

fig_actions.add_bar(
    x=result_no_deg["time_start"],
    y=result_no_deg["charge_kw"],
    name="Charge - no degradation"
)

fig_actions.add_bar(
    x=result_no_deg["time_start"],
    y=-result_no_deg["discharge_kw"],
    name="Discharge - no degradation"
)

fig_actions.update_layout(
    title="Battery actions without degradation",
    xaxis_title="Time",
    yaxis_title="Power (kW)",
    barmode="relative"
)

fig_actions.show()


In [18]:
fig_actions_deg = go.Figure()

fig_actions_deg.add_bar(
    x=result_deg["time_start"],
    y=result_deg["charge_kw"],
    name="Charge - degradation"
)

fig_actions_deg.add_bar(
    x=result_deg["time_start"],
    y=-result_deg["discharge_kw"],
    name="Discharge - degradation"
)

fig_actions_deg.update_layout(
    title="Battery actions with degradation",
    xaxis_title="Time",
    yaxis_title="Power (kW)",
    barmode="relative"
)

fig_actions_deg.show()


## 10. Optional: inspect only active timesteps

Useful for seeing exactly where the optimizer charges or discharges.

In [35]:
print("WITHOUT degradation")
print(result_no_deg["action"].value_counts())
print(
    "Net profit:",
    result_no_deg["cumulative_net_profit_sek"].iloc[-1]
)

print("\nWITH degradation")
print(result_deg["action"].value_counts())
print(
    "Gross trading profit:",
    result_deg["cumulative_trading_profit_sek"].iloc[-1]
)
print(
    "Degradation cost:",
    result_deg["cumulative_degradation_cost_sek"].iloc[-1]
)
print(
    "Net profit:",
    result_deg["cumulative_net_profit_sek"].iloc[-1]
)

WITHOUT degradation
action
hold         46
charge       27
discharge    23
Name: count, dtype: int64
Net profit: 9.922744440048792

WITH degradation
action
hold         63
charge       18
discharge    15
Name: count, dtype: int64
Gross trading profit: 9.501728307296773
Degradation cost: 1.401999999999339
Net profit: 8.09972830729743


In [36]:
fig_economics = go.Figure()

fig_economics.add_trace(
    go.Scatter(
        x=result_deg["time_start"],
        y=result_deg["cumulative_trading_profit_sek"],
        mode="lines",
        name="Gross trading profit"
    )
)

fig_economics.add_trace(
    go.Scatter(
        x=result_deg["time_start"],
        y=result_deg["cumulative_degradation_cost_sek"],
        mode="lines",
        name="Degradation cost"
    )
)

fig_economics.add_trace(
    go.Scatter(
        x=result_deg["time_start"],
        y=result_deg["cumulative_net_profit_sek"],
        mode="lines",
        name="Net profit"
    )
)

fig_economics.update_layout(
    title="Battery economics over time",
    xaxis_title="Time",
    yaxis_title="SEK"
)

fig_economics.show()

In [40]:
def build_prices(
    spot_price,
    supplier_adder=0.0875,
    energy_tax=0.36,
    variable_grid_fee=0.26,
    export_compensation=0.04
):
    buy_price = (
        spot_price
        + supplier_adder
        + energy_tax
        + variable_grid_fee
    )

    sell_price = (
        spot_price
        + export_compensation
    )

    return buy_price, sell_price

In [41]:
buy_price_household, sell_price_household = build_prices(
    prices_df["SEK_per_kWh"].to_numpy()
)

In [44]:
result_household = optimize_battery(
    buy_price=buy_price_household,
    sell_price=sell_price_household,
    capacity=capacity,
    initial_energy=initial_energy,
    min_energy=min_energy,
    max_energy=max_energy,
    max_charge_power=max_charge_power,
    max_discharge_power=max_discharge_power,
    eta_charge=eta_charge,
    eta_discharge=eta_discharge,
    dt=dt,
    degradation_cost=0.20
)

In [45]:
result_household["time_start"] = pd.to_datetime(prices_df["time_start"])
result_household["time_end"] = pd.to_datetime(prices_df["time_end"])

In [46]:
print(result_household["action"].value_counts())

print(
    "Gross trading profit:",
    result_household["cumulative_trading_profit_sek"].iloc[-1]
)

print(
    "Degradation cost:",
    result_household["cumulative_degradation_cost_sek"].iloc[-1]
)

print(
    "Net profit:",
    result_household["cumulative_net_profit_sek"].iloc[-1]
)

action
hold         78
charge       10
discharge     8
Name: count, dtype: int64
Gross trading profit: 5.1622843684146185
Degradation cost: 0.7599999999995439
Net profit: 4.402284368415073


In [47]:
def optimize_household_battery(
    buy_price,
    sell_price,
    load_kw,
    capacity,
    initial_energy,
    min_energy,
    max_energy,
    max_charge_power,
    max_discharge_power,
    eta_charge,
    eta_discharge,
    dt,
    degradation_cost=0.0
):
    T = len(buy_price)

    if not (len(sell_price) == len(load_kw) == T):
        raise ValueError("buy_price, sell_price and load_kw must have same length")

    model = pulp.LpProblem(
        "Household_Battery_Optimizer",
        pulp.LpMinimize
    )

    charge = pulp.LpVariable.dicts(
        "charge",
        range(T),
        lowBound=0,
        upBound=max_charge_power
    )

    discharge = pulp.LpVariable.dicts(
        "discharge",
        range(T),
        lowBound=0,
        upBound=max_discharge_power
    )

    energy = pulp.LpVariable.dicts(
        "energy",
        range(T + 1),
        lowBound=min_energy,
        upBound=max_energy
    )

    grid_import = pulp.LpVariable.dicts(
        "grid_import",
        range(T),
        lowBound=0
    )

    grid_export = pulp.LpVariable.dicts(
        "grid_export",
        range(T),
        lowBound=0
    )

    # Objective:
    # minimize import cost - export revenue + degradation cost
    model += pulp.lpSum(
        (
            buy_price[t] * grid_import[t]
            - sell_price[t] * grid_export[t]
            + degradation_cost * discharge[t]
        ) * dt
        for t in range(T)
    )

    # Initial battery energy
    model += energy[0] == initial_energy

    for t in range(T):

        # Battery dynamics
        model += (
            energy[t + 1]
            ==
            energy[t]
            + eta_charge * charge[t] * dt
            - discharge[t] * dt / eta_discharge
        )

        # Household power balance
        model += (
            grid_import[t]
            + discharge[t]
            ==
            load_kw[t]
            + charge[t]
            + grid_export[t]
        )

    # Terminal condition
    model += energy[T] == initial_energy

    # Solve
    model.solve(pulp.GLPK_CMD(msg=False))

    if pulp.LpStatus[model.status] != "Optimal":
        raise RuntimeError(
            f"Solver status: {pulp.LpStatus[model.status]}"
        )

    rows = []

    cumulative_grid_cost = 0.0
    cumulative_export_revenue = 0.0
    cumulative_degradation_cost = 0.0
    cumulative_net_cost = 0.0

    for t in range(T):
        charge_t = charge[t].value()
        discharge_t = discharge[t].value()
        energy_t = energy[t].value()
        energy_after_t = energy[t + 1].value()

        grid_import_t = grid_import[t].value()
        grid_export_t = grid_export[t].value()

        grid_cost_t = (
            buy_price[t] * grid_import_t * dt
        )

        export_revenue_t = (
            sell_price[t] * grid_export_t * dt
        )

        degradation_cost_t = (
            degradation_cost * discharge_t * dt
        )

        net_cost_t = (
            grid_cost_t
            - export_revenue_t
            + degradation_cost_t
        )

        cumulative_grid_cost += grid_cost_t
        cumulative_export_revenue += export_revenue_t
        cumulative_degradation_cost += degradation_cost_t
        cumulative_net_cost += net_cost_t

        if charge_t > 1e-6:
            action = "charge"
        elif discharge_t > 1e-6:
            action = "discharge"
        else:
            action = "hold"

        rows.append({
            "time": t,

            "buy_price": buy_price[t],
            "sell_price": sell_price[t],
            "load_kw": load_kw[t],

            "grid_import_kw": grid_import_t,
            "grid_export_kw": grid_export_t,

            "charge_kw": charge_t,
            "discharge_kw": discharge_t,

            "energy_kwh": energy_t,
            "energy_after_kwh": energy_after_t,
            "soc": energy_t / capacity,
            "soc_after": energy_after_t / capacity,

            "action": action,

            "grid_cost_sek": grid_cost_t,
            "export_revenue_sek": export_revenue_t,
            "degradation_cost_sek": degradation_cost_t,
            "net_cost_sek": net_cost_t,

            "cumulative_grid_cost_sek": cumulative_grid_cost,
            "cumulative_export_revenue_sek": cumulative_export_revenue,
            "cumulative_degradation_cost_sek": cumulative_degradation_cost,
            "cumulative_net_cost_sek": cumulative_net_cost
        })

    return pd.DataFrame(rows)

In [48]:
import numpy as np

T = 96
dt = 0.25

hours = np.arange(T) * dt

load_kw = (
    0.6
    + 1.2 * np.exp(-((hours - 7.5) / 1.5) ** 2)
    + 2.0 * np.exp(-((hours - 18.5) / 2.0) ** 2)
)

In [49]:
result_v2 = optimize_household_battery(
    buy_price=buy_price_household,
    sell_price=sell_price_household,
    load_kw=load_kw,
    capacity=capacity,
    initial_energy=initial_energy,
    min_energy=min_energy,
    max_energy=max_energy,
    max_charge_power=max_charge_power,
    max_discharge_power=max_discharge_power,
    eta_charge=eta_charge,
    eta_discharge=eta_discharge,
    dt=dt,
    degradation_cost=0.20
)

In [50]:
result_v2["time_start"] = pd.to_datetime(prices_df["time_start"])
result_v2["time_end"] = pd.to_datetime(prices_df["time_end"])

In [51]:
result_v2[
    [
        "time_start",
        "buy_price",
        "load_kw",
        "grid_import_kw",
        "grid_export_kw",
        "charge_kw",
        "discharge_kw",
        "soc",
        "action",
        "net_cost_sek"
    ]
]

,time_start,buy_price,load_kw,grid_import_kw,grid_export_kw,charge_kw,discharge_kw,soc,action,net_cost_sek
0,2026-09-14 00:00:00+02:00,2.02705,0.600000,0.600000,0.0,0.000000,0.0,0.50000,hold,0.304058
1,2026-09-14 00:15:00+02:00,2.02986,0.600000,0.600000,0.0,0.000000,0.0,0.50000,hold,0.304479
2,2026-09-14 00:30:00+02:00,2.03144,0.600000,0.600000,0.0,0.000000,0.0,0.50000,hold,0.304716
3,2026-09-14 00:45:00+02:00,2.03200,0.600000,0.600000,0.0,0.000000,0.0,0.50000,hold,0.304800
4,2026-09-14 01:00:00+02:00,2.03166,0.600000,0.600000,0.0,0.000000,0.0,0.50000,hold,0.304749
...,...,...,...,...,...,...,...,...,...,...
91,2026-09-14 22:45:00+02:00,1.77176,0.621874,1.042926,0.0,0.421053,0.0,0.00000,charge,0.461954
92,2026-09-14 23:00:00+02:00,1.72360,0.612659,2.612659,0.0,2.000000,0.0,0.02500,charge,1.125795
93,2026-09-14 23:15:00+02:00,1.61908,0.607101,2.607101,0.0,2.000000,0.0,0.14375,charge,1.055276
94,2026-09-14 23:30:00+02:00,1.50375,0.603861,2.603861,0.0,2.000000,0.0,0.26250,charge,0.978889


In [52]:
baseline_cost_t = (
    result_v2["buy_price"]
    * result_v2["load_kw"]
    * dt
)

result_v2["baseline_cost_sek"] = baseline_cost_t

result_v2["cumulative_baseline_cost_sek"] = (
    result_v2["baseline_cost_sek"].cumsum()
)

result_v2["cumulative_savings_sek"] = (
    result_v2["cumulative_baseline_cost_sek"]
    - result_v2["cumulative_net_cost_sek"]
)

In [53]:
baseline_total = result_v2["cumulative_baseline_cost_sek"].iloc[-1]
optimized_total = result_v2["cumulative_net_cost_sek"].iloc[-1]
savings_total = result_v2["cumulative_savings_sek"].iloc[-1]

print("Cost without battery:", baseline_total)
print("Cost with optimized battery:", optimized_total)
print("Savings:", savings_total)

Cost without battery: 63.748523686212856
Cost with optimized battery: 56.35725597362017
Savings: 7.391267712592686


In [54]:
fig_cost = go.Figure()

fig_cost.add_trace(
    go.Scatter(
        x=result_v2["time_start"],
        y=result_v2["cumulative_baseline_cost_sek"],
        mode="lines",
        name="Without battery"
    )
)

fig_cost.add_trace(
    go.Scatter(
        x=result_v2["time_start"],
        y=result_v2["cumulative_net_cost_sek"],
        mode="lines",
        name="Optimized battery"
    )
)

fig_cost.update_layout(
    title="Cumulative household electricity cost",
    xaxis_title="Time",
    yaxis_title="SEK"
)

fig_cost.show()

In [55]:
fig_savings = px.line(
    result_v2,
    x="time_start",
    y="cumulative_savings_sek",
    title="Cumulative battery savings"
)

fig_savings.show()